In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
import os
from medmnist import PathMNIST
from tqdm import tqdm
import torch.nn.functional as F
from torchvision.models import inception_v3
from torchvision.transforms import functional as TF
from scipy import linalg

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
num_epochs = 50
batch_size = 64
lr = 0.0002  # Standard DCGAN learning rate
beta1 = 0.5  # Beta1 for Adam optimizer (DCGAN recommendation)
z_dim = 100
image_channels = 3
image_size = 28

# Data loading and preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

train_dataset = PathMNIST(split='train', transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

Using downloaded and verified file: C:\Users\Sudhanshu\.medmnist\pathmnist.npz


In [3]:
# Generator
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Input: z_dim x 1 x 1
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            # 512 x 4 x 4
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # 256 x 8 x 8
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # 128 x 16 x 16
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # 64 x 32 x 32 (slightly larger, crop to 28x28 later if needed)
            nn.ConvTranspose2d(64, image_channels, 3, 1, 1, bias=False),
            nn.Tanh()  # Output: 3 x 28 x 28
        )

    def forward(self, x):
        img = self.main(x)
        # Crop to 28x28 if needed (due to stride/padding)
        return img[:, :, :28, :28]

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # Input: 3 x 28 x 28
            nn.Conv2d(image_channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 64 x 14 x 14
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 128 x 7 x 7
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 256 x 3 x 3
            nn.Conv2d(256, 512, 3, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # 512 x 1 x 1
            nn.Conv2d(512, 1, 1, 1, 0, bias=False),
            nn.Sigmoid()  # Output: 1 (probability)
        )

    def forward(self, x):
        return self.main(x).view(-1, 1)

In [4]:
# Weight initialization (DCGAN recommendation)
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Visualization function
def visualize(real_images, generated_images, epoch):
    real_images = (real_images + 1) / 2  # Denormalize
    generated_images = (generated_images + 1) / 2
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for i in range(5):
        axes[0, i].imshow(real_images[i].permute(1, 2, 0).cpu().numpy())
        axes[0, i].axis('off')
        axes[1, i].imshow(generated_images[i].permute(1, 2, 0).cpu().numpy())
        axes[1, i].axis('off')
    plt.suptitle(f'DCGAN - Epoch {epoch+1}')
    plt.savefig(f'generated_images_DCGAN/epoch_{epoch+1}_comparison.png')
    plt.close()

In [5]:
def load_inception_model(device):
    inception_model = inception_v3(weights='Inception_V3_Weights.IMAGENET1K_V1').to(device)
    inception_model.eval()
    return inception_model

def get_inception_activations(images, inception_model, device, batch_size=32):
    activations = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i + batch_size].to(device)
            batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
            act = inception_model(batch)
            activations.append(act.cpu().numpy())
    return np.concatenate(activations, axis=0)

def compute_fid(real_images, fake_images, inception_model, device, batch_size=32):
    real_acts = get_inception_activations(real_images, inception_model, device, batch_size)
    fake_acts = get_inception_activations(fake_images, inception_model, device, batch_size)
    mu_real, sigma_real = np.mean(real_acts, axis=0), np.cov(real_acts, rowvar=False)
    mu_fake, sigma_fake = np.mean(fake_acts, axis=0), np.cov(fake_acts, rowvar=False)
    diff = mu_real - mu_fake
    covmean = linalg.sqrtm(sigma_real.dot(sigma_fake), disp=False)[0]
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

def compute_inception_score(images, inception_model, device, batch_size=32, splits=10):
    preds = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i + batch_size].to(device)
            batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
            pred = inception_model(batch)
            pred = F.softmax(pred, dim=1).cpu().numpy()
            preds.append(pred)
    preds = np.concatenate(preds, axis=0)
    scores = []
    for i in range(splits):
        part = preds[(i * preds.shape[0] // splits):((i + 1) * preds.shape[0] // splits)]
        kl = part * (np.log(part) - np.log(np.mean(part, axis=0, keepdims=True)))
        kl = np.mean(np.sum(kl, axis=1))
        scores.append(np.exp(kl))
    return np.mean(scores), np.std(scores)

def evaluate(generator, train_loader, device, num_samples=5000, z_dim=100, batch_size=32):
    inception_model = load_inception_model(device)
    generator.eval()
    fake_images = []
    with torch.no_grad():
        for _ in range(num_samples // batch_size):
            z = torch.randn(batch_size, z_dim, 1, 1).to(device)
            fake = generator(z)
            fake_images.append(fake.cpu())
    fake_images = torch.cat(fake_images, dim=0)[:num_samples]
    
    real_images = []
    for batch in train_loader:
        real = batch[0]
        real_images.append(real)
        if len(real_images) * batch_size >= num_samples:
            break
    real_images = torch.cat(real_images, dim=0)[:num_samples]
    
    fid_score = compute_fid(real_images, fake_images, inception_model, device, batch_size)
    is_mean, is_std = compute_inception_score(fake_images, inception_model, device, batch_size)
    return fid_score, is_mean, is_std

In [6]:
# Training function (Fixed)
def train_dcgan():
    generator = Generator(z_dim=z_dim).to(device)
    discriminator = Discriminator().to(device)
    
    generator.apply(weights_init)
    discriminator.apply(weights_init)
    
    criterion = nn.BCELoss()
    optimizer_g = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
    optimizer_d = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))
    
    writer = SummaryWriter('runs/dcgan_experiment')
    os.makedirs('generated_images_DCGAN', exist_ok=True)
    
    fixed_z = torch.randn(5, z_dim, 1, 1).to(device)
    
    real_label = 1.
    fake_label = 0.
    
    for epoch in range(num_epochs):
        generator.train()
        discriminator.train()
        d_loss_total = 0.0
        g_loss_total = 0.0
        
        for i, (real_images, _) in enumerate(tqdm(train_loader, desc=f"DCGAN Epoch {epoch+1}/{num_epochs}")):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)
            
            # Train Discriminator
            discriminator.zero_grad()
            label = torch.full((batch_size,), real_label, dtype=torch.float, device=device)
            output = discriminator(real_images).squeeze(1)  # Squeeze to match label shape
            d_loss_real = criterion(output, label)
            d_loss_real.backward()
            
            z = torch.randn(batch_size, z_dim, 1, 1).to(device)
            fake_images = generator(z)
            label.fill_(fake_label)
            output = discriminator(fake_images.detach()).squeeze(1)  # Squeeze here too
            d_loss_fake = criterion(output, label)
            d_loss_fake.backward()
            optimizer_d.step()
            
            d_loss = d_loss_real + d_loss_fake
            d_loss_total += d_loss.item()
            
            # Train Generator
            generator.zero_grad()
            label.fill_(real_label)
            output = discriminator(fake_images).squeeze(1)  # Squeeze for generator loss
            g_loss = criterion(output, label)
            g_loss.backward()
            optimizer_g.step()
            
            g_loss_total += g_loss.item()
        
        d_loss_avg = d_loss_total / len(train_loader)
        g_loss_avg = g_loss_total / len(train_loader)
        
        writer.add_scalar('Loss/Discriminator', d_loss_avg, epoch)
        writer.add_scalar('Loss/Generator', g_loss_avg, epoch)
        
        print(f"DCGAN Epoch {epoch+1}: d_loss={d_loss_avg:.4f}, g_loss={g_loss_avg:.4f}")
        
        if (epoch + 1) % 5 == 0:
            with torch.no_grad():
                fake_images = generator(fixed_z)
                visualize(real_images[:5], fake_images, epoch)
                real_grid = vutils.make_grid((real_images[:5] + 1) / 2, nrow=5, normalize=False)
                fake_grid = vutils.make_grid((fake_images + 1) / 2, nrow=5, normalize=False)
                writer.add_image('Images/Real', real_grid, epoch)
                writer.add_image('Images/Generated', fake_grid, epoch)
        
        if (epoch + 1) % 10 == 0:
            fid_score, is_mean, is_std = evaluate(generator, train_loader, device, num_samples=5000, z_dim=z_dim)
            writer.add_scalar('Metrics/FID', fid_score, epoch)
            writer.add_scalar('Metrics/Inception Score Mean', is_mean, epoch)
            writer.add_scalar('Metrics/Inception Score Std', is_std, epoch)
            print(f"Epoch {epoch+1} - FID: {fid_score:.2f}, IS: {is_mean:.2f} ± {is_std:.2f}")
    
    print("Training completed. Performing final evaluation...")
    fid_score, is_mean, is_std = evaluate(generator, train_loader, device, num_samples=5000, z_dim=z_dim)
    print(f"Final FID Score: {fid_score:.2f}")
    print(f"Final Inception Score: {is_mean:.2f} ± {is_std:.2f}")
    
    writer.add_scalar('Metrics/FID', fid_score, num_epochs)
    writer.add_scalar('Metrics/Inception Score Mean', is_mean, num_epochs)
    writer.add_scalar('Metrics/Inception Score Std', is_std, num_epochs)
    writer.close()
    
    torch.save(generator.state_dict(), 'final_generator_dcgan.pth')

In [7]:
if __name__ == "__main__":
    train_dcgan()

DCGAN Epoch 1/50: 100%|██████████| 1406/1406 [00:30<00:00, 45.98it/s]


DCGAN Epoch 1: d_loss=0.9911, g_loss=1.2368


DCGAN Epoch 2/50: 100%|██████████| 1406/1406 [00:29<00:00, 47.24it/s]


DCGAN Epoch 2: d_loss=0.1850, g_loss=2.6755


DCGAN Epoch 3/50: 100%|██████████| 1406/1406 [00:30<00:00, 45.84it/s]


...

DCGAN Epoch 48: d_loss=0.7722, g_loss=2.0658


DCGAN Epoch 49/50: 100%|██████████| 1406/1406 [00:38<00:00, 36.17it/s]


DCGAN Epoch 49: d_loss=0.7699, g_loss=2.0600


DCGAN Epoch 50/50: 100%|██████████| 1406/1406 [00:37<00:00, 37.90it/s]


DCGAN Epoch 50: d_loss=0.7635, g_loss=2.0536
Epoch 50 - FID: 53.96, IS: 1.83 ± 0.04
Training completed. Performing final evaluation...
Final FID Score: 51.02
Final Inception Score: 1.83 ± 0.03
